In [1]:
import os
import requests
import numpy as np
from time import sleep
from dotenv import load_dotenv
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

d:\Github\movie-discovery-app\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
load_dotenv()
TMDB_API_KEY = os.getenv("TMDB_API_KEY")
TMDB_URL = "https://api.themoviedb.org/3/"

MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
model = SentenceTransformer(MODEL_NAME)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 804.69it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [3]:
def get_user_input():
    inputs = []
    print("Enter 2–5 words or phrases (type 'done' to finish):")

    while len(inputs) < 5:
        text = input("> ").strip()
        if text.lower() == "done":
            break
        if text:
            inputs.append(text)

    if len(inputs) < 2:
        raise ValueError("Please enter at least 2 inputs.")

    return inputs

In [ ]:
def fetch_movies(pages=3):
    movies = []

    for page in range(1, pages + 1):
        response = requests.get(
            TMDB_URL+"discover/movie",
            params={
                "api_key": TMDB_API_KEY,
                "language": "en-US",
                "sort_by": "vote_count.desc",
                "page": page,
            },
        )
        data = response.json()
        for m in data.get("results", []):
            if m.get("overview"):
                movies.append({
                    "id": m["id"],
                    "title": m["title"],
                    "year": m["release_date"][:4] if m.get("release_date") else "N/A",
                    "overview": m["overview"]
                    "keywords": []
                })

    return movies

# Different Movie queries to be added to the database.

In [ ]:
def fetch_keywords(movies):
    for movie in movies:
        response = requests.get(
            TMDB_URL+f"movie/{movie['id']}/keywords",
            params={"api_key": TMDB_API_KEY}
        )
        data = response.json()
        movie["keywords"] = [k["name"] for k in data.get("keywords", [])]
        sleep(0.1)

In [ ]:
def movie_keyword_text(movie):
    return ', '.join(movie.get("keywords", []))

def movie_overview_text(movie):
    return movie.get("overview", "")

def movie_text(movie):
    keywords = ", ".join(movie.get("keywords", []))
    return f"{keywords}. {movie['overview']}"

In [7]:
movies = fetch_movies(pages=15)
fetch_keywords(movies)

In [ ]:
#movie_texts = [movie_keyword_text(m) for m in movies]
#movie_texts += [movie_overview_text(m) for m in movies]
# movie_texts = [movie_keyword_text(m) + ", " + movie_overview_text(m) for m in movies]
movie_texts = [movie_text(m) for m in movies]

movie_embeddings = model.encode(
    movie_texts,
    normalize_embeddings=True,
    show_progress_bar=True
)

Batches: 100%|██████████| 10/10 [00:03<00:00,  2.76it/s]


In [27]:
movie_texts

['rescue, future, spacecraft, race against time, artificial intelligence (a.i.), nasa, time warp, dystopia, expedition, space travel, wormhole, famine, hibernation, black hole, quantum mechanics, family relationships, space, robot, astronaut, scientist, single father, farmer, space station, space adventure, time paradox, time-manipulation, cryonics, father daughter relationship, 2060s, cornfield, The adventures of a group of explorers who make use of a newly discovered wormhole to surpass the limitations on human space travel and conquer the vast distances involved in an interstellar voyage.',
 'mission, dreams, kidnapping, spy, allegory, intellectual, manipulation, heist, memory, dream world, subconscious, complex, high concept, dream, Cobb, a skilled thief who commits corporate espionage by infiltrating the subconscious of his targets is offered a chance to regain his old life as payment for a task considered to be impossible: "inception", the implantation of another person\'s idea i

# Trying different input queries

In [31]:
user_inputs = get_user_input()
query = ", ".join(user_inputs)

query_embedding = model.encode(
    query,
    normalize_embeddings=True
)

Enter 2–5 words or phrases (type 'done' to finish):


In [39]:
similarities = cosine_similarity(
    [query_embedding],
    movie_embeddings
)[0]

top_indices = np.argsort(similarities)[::-1][:5]

In [ ]:
for rank, idx in enumerate(top_indices, start=1):
    movie = movies[idx]
    score = similarities[idx]
    print(f"{rank}. {movie['title']} ({movie['year']}) (score: {score:.3f})")
    print(f"   Keywords: {', '.join(movie.get('keywords', []))}")
    print(f"   {movie['overview']}\n")

1. Avatar: The Way of Water (2022) (score: 0.410)
   Keywords: dying and death, loss of loved one, alien life-form, resurrection, dysfunctional family, sequel, alien planet, distant future, adopted child, rebirth, family dynamics, adopted son, stronger villain, war, exhilarated, modest
   Set more than a decade after the events of the first film, learn the story of the Sully family (Jake, Neytiri, and their kids), the trouble that follows them, the lengths they go to keep each other safe, the battles they fight to stay alive, and the tragedies they endure.

2. Silver Linings Playbook (2012) (score: 0.353)
   Keywords: depression, infidelity, dancing, friendship, philadelphia, pennsylvania, based on novel or book, widow, superstition, letter, dance competition, teacher, mental institution, therapy, mental illness, ex-wife, institutionalization, death of husband, unfaithful wife, father son relationship
   After losing his job and wife, and spending time in an institution, a former teach

: 

# Emotion Reccomender

In [34]:
EMOTION_MAP = {
    "mad": "wholesome calming uplifting feel-good movie",
    "sad": "happy lighthearted comedy warm funny movie",
    "short-tempered": "fast-paced energetic short fun movie",
    "anxious": "comforting grounded reassuring gentle movie",
    "well": "serious thoughtful emotional dramatic movie"
}

def get_emotion_input():
    print("\nHow are you feeling?")
    print("Options:", ", ".join(EMOTION_MAP.keys()))
    emotion = input("> ").strip().lower()

    if emotion not in EMOTION_MAP:
        raise ValueError("Unsupported emotion.")

    return emotion

In [38]:
emotion = get_emotion_input()
emotion_text = EMOTION_MAP[emotion]
query_embedding = model.encode(emotion_text, normalize_embeddings=True)

print("\nUse opposite vector? (y/n)")
if input("> ").strip().lower() == "y":
    query_embedding = -query_embedding


How are you feeling?
Options: mad, sad, short-tempered, anxious, well

Use opposite vector? (y/n)
